# 关于异步事件 更加底层 的东西

- ___四个核心基本概念___
> 1. 协程
> 2. task
> 3. future
> 4. 事件循环loop

- 单线程高并发——常见于网络请求，生产消费，定时任务等
- `asyncio.run`实际上就是在启动一个异步事件的循环，循环内部有一个`selector`调度器，负责调动 task（task是基本调度单位），下面逐一拆解这四个概念


- jupyter 在执行时已经启动了一个事件循环了，这里为了演示需要嵌套 异步循环，所以用到了 nest_asyncio
- 在jupyter 内部已经有一个异步事件循环了，因此对于协程，直接使用await就能执行了（在python脚本里面是不行的）
- 这里写了一个基本示例，可以辅助理解（结合下面这段代码来看）：
> 1. work() 返回一个协程对象。
> 2. asyncio.create_task(work()) 创建 Task，并把 Task 注册到事件循环。
> 3. 事件循环调度 Task，Task 调用 coro.send(None) 推进协程。
> 4. 协程打印 work 开始，然后执行 await asyncio.sleep(1)。
> 5. asyncio.sleep(1) 内部创建一个 Future，并告诉事件循环：“1 秒后调用 future.set_result(None)。”
> 6. Task 发现协程 yield 了这个 Future，于是把“恢复自己”的回调注册到 Future 上，然后 Task 挂起，控制权回到事件循环。
> 7. 事件循环继续运行其他任务，或者等待 I/O。
> 8. 1 秒后，定时器到期，事件循环执行回调，调用 future.set_result(None)。
> 9. Future 完成，事件循环调度之前注册的“恢复 Task”的回调。
> 10. Task 再次调用 coro.send(None)，协程从 await 处恢复，打印 work 结束，返回 42。
> 11. Task 自动调用 task.set_result(42)。
> 12. await task 的地方拿到结果 42。

In [6]:
import  nest_asyncio
from zmq.eventloop import future

nest_asyncio.apply()

import asyncio

In [7]:
async def main():
    await asyncio.sleep(1)
    print("main 完成")
    return 42

loop = asyncio.new_event_loop()      # 1. 创建一个新的事件循环
asyncio.set_event_loop(loop)         # 2. 设为当前线程的默认循环（可选，但兼容旧代码时有用）
try:
    result = loop.run_until_complete(main())   # 3. 运行协程直到完成
    print("结果:", result)
finally:
    loop.run_until_complete(loop.shutdown_asyncgens())  # 4. 关闭异步生成器
    loop.run_until_complete(loop.shutdown_default_executor())  # 5. 关闭默认线程池（3.9+）
    asyncio.set_event_loop(None)     # 6. 解除绑定
    loop.close()                     # 7. 关闭循环，释放底层资源

main 完成
结果: 42


## task 和 协程 和 future
> 1. 异步事件循环调度的基本单位——task
> 2. 协程经过包装变成 task
> 3. 协程是“要执行的异步逻辑”，Future 是“未来结果的容器”，Task 是“把协程交给事件循环执行，并把结果放进 Future 的包装器”。Task 是 Future 的子类。

- 协程
> 1. 协程（Coroutine）是一种可以在执行过程中暂停、再恢复的函数
> 2. 协程在执行过程中可以让出控制权，等之后从上次执行的地方继续
> 3. 协程的切换由程序自己控制，不由操作系统控制
> 4. 写 `await x` 时，Python 会先取得对象 x 的可等待`（awaitable）对象（包括task，future，coroutine都是可等待对象）`
> 5. 底层其实是生成器，可以把 `await` 理解成 `yield from`, 本质是挂起当前协程并把控制权交还事件循环，而不是阻塞线程

- task
> 1. 有多种创建方式，这里只介绍最常见的几种方式
> 2. 是异步事件循环调度的基本单位

In [33]:
async def wakeup(name):
    print(f"loop {id(asyncio.get_event_loop())} is the working loop")
    await asyncio.sleep(1)
    print(f"{name} wake up")

async def work(name):
    await wakeup(name)
    print(f"{name} is working")

In [24]:
# 直接把task1 绑定到当前正在执行的事件循环上
task1 = asyncio.create_task(work("Alice"))
await task1

loop 1480940733552 is the working loop
Alice wake up
Alice is working


In [25]:
# 自定义创建一个新的事件循环，然后把 task 绑到该循环上
my_loop = asyncio.new_event_loop()
task2 = my_loop.create_task(work("Bob"))
asyncio.set_event_loop(my_loop)
my_loop.run_until_complete(task2)

loop 1480941890176 is the working loop
Bob wake up
Bob is working


In [26]:
# gather() 接收多个协程或 awaitable，会自动把它们包装成 Task 并发执行，并返回一个汇总的 Future，结果按传入顺序排列
result = await asyncio.gather(work("Alice"), work("Bob"))
result

loop 1480940733552 is the working loop
loop 1480940733552 is the working loop
Alice wake up
Alice is working
Bob wake up
Bob is working


[None, None]

- future
> 1. Future = “一个现在还没有结果，但未来会有结果的对象”
> 2. 三种状态 `PENDING`, `CANCELLED`, `FINISHED`,
> 3. 状态查询函数 `done()`,`cancelled()`,`result()`,`exception()`

In [39]:
import asyncio

async def main():
    loop = asyncio.get_running_loop()
    fut = loop.create_future()

    async def setter():
        await asyncio.sleep(1)
        # 下面几种情况都可以尝试一下
        # fut.set_result(42)
        # fut.set_exception(ValueError("出错了"))
        fut.cancel()

    asyncio.create_task(setter())

    try:
        result = await fut
        print("得到结果:", result)
    except ValueError as e:
        print("捕获异常:", e)
    except asyncio.CancelledError:
        print("future 被取消了")

asyncio.run(main())

future 被取消了


In [41]:
import asyncio
from functools import partial


def on_done(fut, name):
    print(f"[{name}] 回调触发")
    if fut.cancelled():
        print(f"[{name}] Future 被取消了")
    elif fut.exception() is not None:
        print(f"[{name}] Future 异常: {fut.exception()}")
    else:
        print(f"[{name}] Future 结果: {fut.result()}")


async def main():
    loop = asyncio.get_running_loop()
    fut = loop.create_future()

    # 注册回调，注意传递可调用对象，用 partial 绑定额外参数
    fut.add_done_callback(partial(on_done, name="A"))
    fut.add_done_callback(partial(on_done, name="B"))

    async def setter():
        await asyncio.sleep(1)
        # 设置结果，触发所有回调
        fut.set_result("hello")

    asyncio.create_task(setter())

    try:
        result = await fut
        print("await 得到结果:", result)
    except asyncio.CancelledError:
        print("await 发现 future 被取消了")
    except Exception as e:
        print("await 捕获异常:", e)


asyncio.run(main())

[A] 回调触发
[A] Future 结果: hello
[B] 回调触发
[B] Future 结果: hello
await 得到结果: hello


## 事件循环
> 1. 负责决定：现在该运行哪个回调、哪个 Task、哪个 Future 的回调，以及该等哪个 I/O 或定时器
> 2. 事件循环不直接执行协程，它执行的是回调；Task 把协程翻译成回调，Future 是挂起和恢复的开关

- 一次性执行——`run_until_complete()`会不断循环迭代直到事件循环里面注册的task执行结束

In [27]:
import asyncio

async def fetch(name, delay):
    print(f"{name} 开始")
    await asyncio.sleep(delay)
    print(f"{name} 完成")
    return name

my_loop = asyncio.new_event_loop()   # 创建一个新的事件循环
asyncio.set_event_loop(my_loop)      # 设为当前线程的默认循环（可选）

try:
    result = my_loop.run_until_complete(fetch("A", 1))
    print(result)
finally:
    loop.close()                  # 手动关闭

A 开始
A 完成
A


- 逐步式推进——run_once 式的控制
> 1. `loop.call_soon(callback, *args, context=None)` 安排 callback 在下一次事件循环迭代中尽快执行（当前回调执行完后，尽快执行）
> 2. `loop.call_later(delay, callback, *args, context=None)` 在 delay 秒后执行 callback
> 3. `loop.call_at(when, callback, *args, context=None)` 在事件循环的绝对时间 when 执行 callback

In [31]:
import asyncio

def hello(name):
    print(f"hello {name}")

async def main():
    loop = asyncio.get_running_loop()
    print("main 开始")
    loop.call_soon(hello, "Alice")
    await asyncio.sleep(0.1)
    print("main 结束")

asyncio.run(main())

main 开始
hello Alice
main 结束


In [28]:
import asyncio

async def tick(n):
    for i in range(n):
        print(f"tick {i}")
        await asyncio.sleep(0.5)

loop = asyncio.new_event_loop()
task = loop.create_task(tick(10))

loop.call_later(2.2, loop.stop)   # 2.2 秒后请求停止循环
loop.run_forever()                # 持续运行，直到 stop() 被调用

print("循环已停止，任务还未完成:", not task.done())
loop.close()

tick 0
tick 1
tick 2
tick 3
tick 4
循环已停止，任务还未完成: True


In [29]:
import asyncio
loop = asyncio.new_event_loop()
async def tick(n):
    global loop
    for i in range(n):
        if i == 4:
            loop.stop()
        print(f"tick {i}")
        await asyncio.sleep(0.5)


task = loop.create_task(tick(10))

loop.run_forever()                # 持续运行，直到 loop.stop() 被调用

print("循环已停止，任务还未完成:", not task.done())
loop.close()

tick 0
tick 1
tick 2
tick 3
tick 4
循环已停止，任务还未完成: True


- 一些常见的用来控制事件循环的方法
> `loop.run_until_complete(future)`	运行循环，直到某个 Future/Task 完成
>
> `loop.run_forever()`	一直运行，直到有人调用 loop.stop()
>
> `loop.stop()`	停止循环
>
> `loop.close()`	关闭循环，之后不能再用
>
> `loop.call_soon(cb)`	尽快调度回调
>
> `loop.call_later(delay, cb)`	延迟调度回调
>
> `loop.call_at(when, cb)`	指定时间调度回调
>
> `loop.create_future()`	创建 Future
>
> `loop.create_task(coro)`	创建 Task
>
> `loop.run_in_executor(...)`	把阻塞函数丢到线程/进程池
>

In [32]:
async def main():
    loop = asyncio.get_running_loop()
    fut = loop.create_future()             # 创建一个 pending 的 Future
    loop.call_later(1, fut.set_result, "数据到了")  # 1 秒后由循环设置结果
    result = await fut                     # 挂起，直到 fut 完成
    print(result)

asyncio.run(main())

数据到了
